# 01 - Baselines & Prophet

Ce notebook etablit des baselines et compare avec Prophet.
**Il utilise exactement les memes patterns que `train.py` pour garantir la coherence.**

## Ameliorations par rapport a la version precedente:
- Utilisation des memes chemins de donnees que train.py
- Nettoyage avec `clean_dataframe()` de src.data.clean
- Split temporel identique a train.py
- Metriques robustes (safe MAPE, sMAPE) identiques a train.py

In [ ]:
import sys
import warnings
from pathlib import Path
from math import sqrt

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

# Setup project path - SAME AS train.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Import project modules - SAME AS train.py
try:
    from src.config import TRAIN_CSV, GROUP_COLS
    print(f"Config loaded: TRAIN_CSV={TRAIN_CSV}")
except ImportError:
    print("src.config not found, using defaults")
    TRAIN_CSV = PROJECT_ROOT / "data" / "interim" / "train.csv"
    GROUP_COLS = ['store_id', 'product_id']

try:
    from src.data.clean import clean_dataframe
    CLEAN_AVAILABLE = True
    print("clean_dataframe() available")
except ImportError:
    CLEAN_AVAILABLE = False
    print("clean_dataframe() not available")

## 1. Configuration (identique a train.py)

In [ ]:
# Configuration - SAME AS train.py
HORIZON = 14  # Forecast horizon in days (same default as train.py)
DATE_COL = 'date'
TARGET_COL = 'value'

# Find data file - SAME LOGIC AS train.py load_and_prepare()
DATA_PATHS = [
    TRAIN_CSV,
    PROJECT_ROOT / "data" / "processed" / "uploaded_generated_training_10950_features.csv",
    PROJECT_ROOT / "data" / "processed" / "train_features.csv",
]

DATA_PATH = None
for p in DATA_PATHS:
    if p.exists():
        DATA_PATH = p
        break

if DATA_PATH is None:
    # Fallback: find any CSV
    for folder in ["interim", "processed", "raw"]:
        folder_path = PROJECT_ROOT / "data" / folder
        if folder_path.exists():
            csvs = list(folder_path.glob("*.csv"))
            if csvs:
                DATA_PATH = csvs[0]
                break

if DATA_PATH is None:
    raise FileNotFoundError("No data file found!")

print(f"Using data: {DATA_PATH}")

## 2. Chargement et Nettoyage (identique a train.py)

In [ ]:
# Load data - SAME AS train.py
df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
print(f"Shape initiale: {df.shape}")
print(f"Colonnes: {list(df.columns)}")
df.head()

In [ ]:
# Apply cleaning - SAME AS train.py load_and_prepare()
if CLEAN_AVAILABLE:
    df = clean_dataframe(
        df, 
        date_col=DATE_COL, 
        value_col=TARGET_COL, 
        fill_strategy='ffill', 
        outlier_threshold=None
    )
    print("clean_dataframe() applique")
else:
    df[DATE_COL] = pd.to_datetime(df[DATE_COL])
    df = df.dropna(subset=[TARGET_COL])
    print("Nettoyage manuel applique")

# Remap columns - SAME AS train.py
if 'store_id' not in df.columns and 'id' in df.columns:
    df = df.rename(columns={'id': 'store_id'})
    print("Remapped 'id' -> 'store_id'")

# Detect available group columns
available_groups = [c for c in GROUP_COLS if c in df.columns]
print(f"Group columns: {available_groups}")

# Sort by group + date - SAME AS train.py
sort_cols = available_groups + [DATE_COL]
df = df.sort_values(sort_cols).reset_index(drop=True)

print(f"\nShape apres nettoyage: {df.shape}")
print(f"Target stats: mean={df[TARGET_COL].mean():.2f}, std={df[TARGET_COL].std():.2f}")
print(f"Target range: [{df[TARGET_COL].min():.2f}, {df[TARGET_COL].max():.2f}]")

In [ ]:
# Cap extreme outliers (Winsorization) - critical for Prophet stability
lower = df[TARGET_COL].quantile(0.005)
upper = df[TARGET_COL].quantile(0.995)

n_clipped = ((df[TARGET_COL] < lower) | (df[TARGET_COL] > upper)).sum()
df[TARGET_COL] = df[TARGET_COL].clip(lower=max(0, lower), upper=upper)

print(f"Valeurs clippees: {n_clipped} ({100*n_clipped/len(df):.2f}%)")
print(f"Nouveau range: [{df[TARGET_COL].min():.2f}, {df[TARGET_COL].max():.2f}]")

## 3. Split Temporel (identique a train.py split_time_series())

In [ ]:
# Temporal split - EXACT SAME LOGIC AS train.py split_time_series()
unique_dates = sorted(df[DATE_COL].unique())
print(f"Dates uniques: {len(unique_dates)}")
print(f"Range: {unique_dates[0]} to {unique_dates[-1]}")

# Adjust horizon if necessary - SAME AS train.py
if HORIZON >= len(unique_dates):
    HORIZON = max(1, len(unique_dates) // 5)
    print(f"Horizon ajuste a {HORIZON}")

cutoff_date = unique_dates[-HORIZON]
train_mask = df[DATE_COL] < cutoff_date
test_mask = df[DATE_COL] >= cutoff_date

df_train = df[train_mask].copy()
df_test = df[test_mask].copy()

print(f"\nCutoff: {cutoff_date}")
print(f"Train: {len(df_train)} samples")
print(f"Test: {len(df_test)} samples")

## 4. Fonctions de Metriques (identiques a train.py evaluate_model())

In [ ]:
def compute_metrics(y_true, y_pred, name="Model"):
    """Compute metrics - SAME LOGIC AS train.py evaluate_model()"""
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    bias = float(np.mean(y_pred - y_true))
    
    # Safe MAPE - SAME AS train.py
    mask = np.abs(y_true) > 1.0
    if mask.sum() > 0:
        mape = float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)
    else:
        mape = None
    
    # sMAPE - SAME AS train.py
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom_safe = np.where(denom > 0, denom, 1.0)
    smape = float(np.mean(np.abs(y_true - y_pred) / denom_safe) * 100)
    
    print(f"\n{name}:")
    print(f"  MAE   = {mae:.4f}")
    print(f"  RMSE  = {rmse:.4f}")
    print(f"  R2    = {r2:.4f}")
    print(f"  Bias  = {bias:.4f}")
    if mape is not None:
        print(f"  MAPE  = {mape:.2f}%")
    print(f"  sMAPE = {smape:.2f}%")
    
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'Bias': bias, 'MAPE': mape, 'sMAPE': smape}

## 5. Baseline 1: Naive (derniere valeur par groupe)

In [ ]:
def naive_forecast(df_train, df_test, group_cols, target_col):
    """Naive: use last known value per group"""
    global_mean = df_train[target_col].mean()
    
    if group_cols:
        last_values = df_train.groupby(group_cols)[target_col].last()
        
        if len(group_cols) == 1:
            predictions = df_test[group_cols[0]].map(last_values.to_dict()).fillna(global_mean)
        else:
            # Multi-column grouping
            df_test_indexed = df_test.set_index(group_cols)
            predictions = df_test_indexed.index.map(
                lambda x: last_values.get(x, global_mean)
            )
            predictions = pd.Series(predictions).fillna(global_mean)
    else:
        last_value = df_train[target_col].iloc[-1]
        predictions = pd.Series([last_value] * len(df_test))
    
    return np.array(predictions)

y_naive = naive_forecast(df_train, df_test, available_groups, TARGET_COL)
metrics_naive = compute_metrics(df_test[TARGET_COL].values, y_naive, "Naive Baseline")

## 6. Baseline 2: Moyenne Mobile (7 jours)

In [ ]:
def ma_forecast(df_train, df_test, group_cols, target_col, window=7):
    """Moving average: use rolling mean per group"""
    global_mean = df_train[target_col].mean()
    
    if group_cols:
        ma_values = df_train.groupby(group_cols)[target_col].apply(
            lambda x: x.tail(window).mean()
        )
        
        if len(group_cols) == 1:
            predictions = df_test[group_cols[0]].map(ma_values.to_dict()).fillna(global_mean)
        else:
            df_test_indexed = df_test.set_index(group_cols)
            predictions = df_test_indexed.index.map(
                lambda x: ma_values.get(x, global_mean)
            )
            predictions = pd.Series(predictions).fillna(global_mean)
    else:
        ma_value = df_train[target_col].tail(window).mean()
        predictions = pd.Series([ma_value] * len(df_test))
    
    return np.array(predictions)

y_ma = ma_forecast(df_train, df_test, available_groups, TARGET_COL, window=7)
metrics_ma = compute_metrics(df_test[TARGET_COL].values, y_ma, "MA(7) Baseline")

## 7. Prophet

In [ ]:
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print("Prophet disponible")
except ImportError:
    PROPHET_AVAILABLE = False
    print("Prophet non installe - pip install prophet")

In [ ]:
if PROPHET_AVAILABLE:
    # Aggregate by date for Prophet (sum of all groups)
    prophet_train = df_train.groupby(DATE_COL)[TARGET_COL].sum().reset_index()
    prophet_train.columns = ['ds', 'y']
    prophet_train = prophet_train.sort_values('ds')
    
    prophet_test = df_test.groupby(DATE_COL)[TARGET_COL].sum().reset_index()
    prophet_test.columns = ['ds', 'y']
    prophet_test = prophet_test.sort_values('ds')
    
    # Ensure no negatives
    prophet_train['y'] = prophet_train['y'].clip(lower=0)
    prophet_test['y'] = prophet_test['y'].clip(lower=0)
    
    print(f"Prophet train: {len(prophet_train)} daily records")
    print(f"Prophet test: {len(prophet_test)} daily records")
    print(f"Target range: [{prophet_train['y'].min():.2f}, {prophet_train['y'].max():.2f}]")

In [ ]:
if PROPHET_AVAILABLE:
    # Determine seasonality mode
    cv = prophet_train['y'].std() / (prophet_train['y'].mean() + 1e-8)
    seasonality_mode = 'multiplicative' if cv > 0.5 else 'additive'
    print(f"CV: {cv:.3f} -> {seasonality_mode} seasonality")
    
    # Configure Prophet
    n_days = len(prophet_train)
    model_prophet = Prophet(
        seasonality_mode=seasonality_mode,
        yearly_seasonality=(n_days >= 365),
        weekly_seasonality=True,
        daily_seasonality=False,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=0.1,
    )
    
    print("\nEntrainement Prophet...")
    model_prophet.fit(prophet_train)
    print("Termine")

In [ ]:
if PROPHET_AVAILABLE:
    # Predict
    future = prophet_test[['ds']].copy()
    forecast = model_prophet.predict(future)
    
    y_prophet = forecast['yhat'].clip(lower=0).values
    y_true_prophet = prophet_test['y'].values
    
    metrics_prophet = compute_metrics(y_true_prophet, y_prophet, "Prophet")

In [ ]:
if PROPHET_AVAILABLE:
    fig = model_prophet.plot_components(forecast)
    plt.tight_layout()
    plt.show()

## 8. Comparaison des Modeles

In [ ]:
results = {
    'Naive': metrics_naive,
    'MA(7)': metrics_ma,
}
if PROPHET_AVAILABLE:
    results['Prophet'] = metrics_prophet

summary_df = pd.DataFrame(results).T[['MAE', 'RMSE', 'R2', 'sMAPE']]

print("\n" + "="*60)
print("COMPARAISON DES BASELINES")
print("="*60)
print(summary_df.round(4).to_string())
print("="*60)

best_model = summary_df['MAE'].idxmin()
print(f"\nMeilleur modele (MAE): {best_model}")

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Time series
ax1 = axes[0, 0]
ts_train = df_train.groupby(DATE_COL)[TARGET_COL].sum()
ts_test = df_test.groupby(DATE_COL)[TARGET_COL].sum()
ax1.plot(ts_train.index, ts_train.values, label='Train', alpha=0.7)
ax1.plot(ts_test.index, ts_test.values, label='Test', linewidth=2)
ax1.axvline(x=cutoff_date, color='red', linestyle='--', label='Cutoff')
ax1.set_title('Serie Temporelle')
ax1.legend()

# 2. Predictions vs Actual
ax2 = axes[0, 1]
if PROPHET_AVAILABLE:
    ax2.scatter(y_true_prophet, y_prophet, alpha=0.6)
    max_val = max(y_true_prophet.max(), y_prophet.max())
    ax2.plot([0, max_val], [0, max_val], 'r--', label='Perfect')
    ax2.set_xlabel('Actual')
    ax2.set_ylabel('Predicted')
    ax2.set_title(f'Prophet (R2={metrics_prophet["R2"]:.3f})')
    ax2.legend()

# 3. MAE/RMSE comparison
ax3 = axes[1, 0]
x = np.arange(2)
width = 0.25
for i, (model, metrics) in enumerate(results.items()):
    ax3.bar(x + i*width, [metrics['MAE'], metrics['RMSE']], width, label=model)
ax3.set_xticks(x + width)
ax3.set_xticklabels(['MAE', 'RMSE'])
ax3.set_title('Erreurs')
ax3.legend()

# 4. R2 comparison
ax4 = axes[1, 1]
r2_values = [results[m]['R2'] for m in results]
colors = ['green' if r2 > 0.5 else 'orange' if r2 > 0 else 'red' for r2 in r2_values]
ax4.bar(list(results.keys()), r2_values, color=colors)
ax4.axhline(y=0.5, color='gray', linestyle='--')
ax4.set_title('R2 Score')
ax4.set_ylabel('R2')

plt.tight_layout()
plt.show()

## 9. Conclusion

In [ ]:
print("\n" + "="*60)
print("RESUME")
print("="*60)
print(f"Dataset: {DATA_PATH.name}")
print(f"Train: {len(df_train)} | Test: {len(df_test)}")
print(f"Horizon: {HORIZON} jours")
print(f"\nMeilleur baseline: {best_model}")
print(f"  MAE   = {results[best_model]['MAE']:.4f}")
print(f"  R2    = {results[best_model]['R2']:.4f}")
print(f"  sMAPE = {results[best_model]['sMAPE']:.2f}%")
print("\nCes baselines servent de reference pour XGBoost/LightGBM.")
print("="*60)